In [1]:
# Find-S & Candidate Elimination | Flight Delay Prediction (Complete Dataset)
# Attributes: Airline, TimeOfDay, DayOfWeek, Season, AirportSize, Weather, PreviousDelay, Congestion

import pandas as pd

def covers(h, x):
    for i in range(len(x)):
        if h[i] != '?' and h[i] != '∅' and h[i] != x[i]:
            return False
    return True

def is_more_general(h1, h2):
    for i in range(len(h1)):
        if h1[i] != '?' and h1[i] != h2[i] and h2[i] != '∅':
            return False
    return True

# ============================================
# DATASET: Flight Delay Prediction
# ============================================

print("="*70)
print("DATASET: Flight Delay Prediction")
print("="*70)

dataset_data = {
    'Sit': [1,1,2,2,3,3,3,3,4,4,4,5,5,5,5],
    'Situation': [
        'Identical diff labels', 'Identical diff labels',
        'Positive ⊏ Negative', 'Positive ⊏ Negative',
        'XOR pattern', 'XOR pattern', 'XOR pattern', 'XOR pattern',
        'Missing attribute', 'Missing attribute', 'Missing attribute',
        'Sequential contradiction', 'Sequential contradiction', 'Sequential contradiction', 'Sequential contradiction'
    ],
    'Airline': ['Delta','Delta','United','United','Delta','Delta','Delta','Delta','Delta','Delta','United','Delta','Delta','United','Delta'],
    'TimeOfDay': ['Evening','Evening','Morning','?','Evening','Morning','Evening','Morning','Evening','Morning','Evening','Morning','Morning','Morning','Morning'],
    'DayOfWeek': ['Weekday','Weekday','Weekday','?','Weekday','Weekday','Weekday','Weekday','Weekday','Weekday','Weekday','Weekday','Weekday','Weekday','Weekday'],
    'Season': ['Summer','Summer','Summer','?','Summer','Summer','Summer','Summer','Summer','Summer','Summer','Summer','Summer','Summer','Summer'],
    'AirportSize': ['Large','Large','Medium','?','Large','Large','Large','Large','Medium','Medium','Medium','Large','Large','Large','Large'],
    'Weather': ['Storm','Storm','Storm','?','Storm','Clear','Storm','Clear','Clear','Clear','Clear','Storm','Rain','Storm','Storm'],
    'PreviousDelay': ['Yes','Yes','No','?','Yes','Yes','Yes','Yes','No','No','No','Yes','No','Yes','Yes'],
    'Congestion': ['High','High','Medium','?','High','Low','Low','High','Medium','Medium','Medium','High','Medium','High','High'],
    'Label': ['Delayed','On Time','Delayed','On Time','Delayed','Delayed','On Time','On Time','Delayed','On Time','On Time','Delayed','On Time','Delayed','On Time']
}

df = pd.DataFrame(dataset_data)
print("\nDataset Overview:")
print(df.to_string(index=False))

print("\n" + "="*70)
print("Attribute Description:")
print("-"*70)
print("| Attribute     | Values                                      |")
print("|---------------|---------------------------------------------|")
print("| Airline       | Delta, United, American, Southwest          |")
print("| TimeOfDay     | Morning, Afternoon, Evening, Night          |")
print("| DayOfWeek     | Weekday, Weekend                            |")
print("| Season        | Summer, Winter, Spring, Fall                |")
print("| AirportSize   | Large, Medium, Small                        |")
print("| Weather       | Clear, Rain, Snow, Storm                    |")
print("| PreviousDelay | Yes, No                                     |")
print("| Congestion    | High, Medium, Low                           |")
print("| Label         | Delayed (Positive) / On Time (Negative)     |")
print("="*70)

DATASET: Flight Delay Prediction

Dataset Overview:
 Sit                Situation Airline TimeOfDay DayOfWeek Season AirportSize Weather PreviousDelay Congestion   Label
   1    Identical diff labels   Delta   Evening   Weekday Summer       Large   Storm           Yes       High Delayed
   1    Identical diff labels   Delta   Evening   Weekday Summer       Large   Storm           Yes       High On Time
   2      Positive ⊏ Negative  United   Morning   Weekday Summer      Medium   Storm            No     Medium Delayed
   2      Positive ⊏ Negative  United         ?         ?      ?           ?       ?             ?          ? On Time
   3              XOR pattern   Delta   Evening   Weekday Summer       Large   Storm           Yes       High Delayed
   3              XOR pattern   Delta   Morning   Weekday Summer       Large   Clear           Yes        Low Delayed
   3              XOR pattern   Delta   Evening   Weekday Summer       Large   Storm           Yes        Low On Time
   3

In [2]:
def find_s(examples, attrs):
    S = ['∅'] * len(attrs)
    for vals, label in examples:
        if label == 'Positive':
            for i in range(len(attrs)):
                if S[i] == '∅':
                    S[i] = vals[i]
                elif S[i] != vals[i]:
                    S[i] = '?'
    return S

In [3]:
def candidate_elimination(examples, attrs):
    S = ['∅'] * len(attrs)
    G = [['?'] * len(attrs)]
    trace = []

    for idx, (vals, label) in enumerate(examples, 1):
        if label == 'Positive':
            for i in range(len(attrs)):
                if S[i] == '∅':
                    S[i] = vals[i]
                elif S[i] != vals[i]:
                    S[i] = '?'
            G = [h for h in G if covers(h, vals)]
        else:
            new_G = []
            for h in G:
                if covers(h, vals):
                    for i in range(len(attrs)):
                        if h[i] == '?':
                            new_h = h.copy()
                            new_h[i] = vals[i]
                            new_G.append(new_h)
                else:
                    new_G.append(h)
            unique = []
            for h in new_G:
                if h not in unique:
                    unique.append(h)
            new_G = unique
            G_temp = []
            for h in new_G:
                is_specific = False
                for other in new_G:
                    if h != other and all(other[i] == '?' or other[i] == h[i] for i in range(len(attrs))):
                        is_specific = True
                        break
                if not is_specific:
                    G_temp.append(h)
            G = [h for h in G_temp if not covers(h, vals)]
            G = [h for h in G if all(S[i] == '∅' or S[i] == '?' or h[i] == '?' or h[i] == S[i] for i in range(len(attrs)))]

        trace.append((idx, vals, label, S.copy(), [h.copy() for h in G]))

        if not G:
            return S, G, trace, False

        for ex_vals, ex_label in examples[:idx]:
            if ex_label == 'Negative' and covers(S, ex_vals):
                return S, G, trace, False

    return S, G, trace, True

In [4]:
print("="*70)
print("SITUATION 1: Identical Examples with Different Labels")
print("="*70)

# Filter dataset for situation 1
sit1 = df[df['Sit'] == 1]
attrs = ['Airline', 'TimeOfDay', 'Weather', 'PreviousDelay']
examples = []

for _, row in sit1.iterrows():
    vals = [row['Airline'], row['TimeOfDay'], row['Weather'], row['PreviousDelay']]
    label = 'Positive' if row['Label'] == 'Delayed' else 'Negative'
    examples.append((vals, label))

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] Same flight (Delta, Evening, Storm, PreviousDelay=Yes) labeled both Delayed and On Time")
print("      → Direct contradiction in training data")

SITUATION 1: Identical Examples with Different Labels

[INPUT]
  E1: {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Positive
  E2: {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Negative

[FIND-S]
  Final S: {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'}

[CANDIDATE ELIMINATION]

  After E1: {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Positive
    S = {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'}
    G = [{'Airline': '?', 'TimeOfDay': '?', 'Weather': '?', 'PreviousDelay': '?'}]

  After E2: {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Negative
    S = {'Airline': 'Delta', 'TimeOfDay': 'Evening', 'Weather': 'Storm', 'PreviousDelay': 'Yes'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] Sa

In [5]:
print("="*70)
print("SITUATION 2: Positive More Specific than Negative")
print("="*70)

# Filter dataset for situation 2
sit2 = df[df['Sit'] == 2]
attrs = ['Airline', 'Weather']
examples = []

for _, row in sit2.iterrows():
    vals = [row['Airline'], row['Weather']]
    label = 'Positive' if row['Label'] == 'Delayed' else 'Negative'
    examples.append((vals, label))

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] Positive (United, Storm) is specialization of negative (United, ?)")
print("      Any hypothesis covering United,Storm also covers United,Clear,Rain,Snow")

SITUATION 2: Positive More Specific than Negative

[INPUT]
  E1: {'Airline': 'United', 'Weather': 'Storm'} -> Positive
  E2: {'Airline': 'United', 'Weather': '?'} -> Negative

[FIND-S]
  Final S: {'Airline': 'United', 'Weather': 'Storm'}

[CANDIDATE ELIMINATION]

  After E1: {'Airline': 'United', 'Weather': 'Storm'} -> Positive
    S = {'Airline': 'United', 'Weather': 'Storm'}
    G = [{'Airline': '?', 'Weather': '?'}]

  After E2: {'Airline': 'United', 'Weather': '?'} -> Negative
    S = {'Airline': 'United', 'Weather': 'Storm'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] Positive (United, Storm) is specialization of negative (United, ?)
      Any hypothesis covering United,Storm also covers United,Clear,Rain,Snow


In [6]:
print("="*70)
print("SITUATION 3: XOR Pattern")
print("="*70)

# Filter dataset for situation 3
sit3 = df[df['Sit'] == 3]
attrs = ['Weather', 'Congestion']
examples = []

for _, row in sit3.iterrows():
    vals = [row['Weather'], row['Congestion']]
    label = 'Positive' if row['Label'] == 'Delayed' else 'Negative'
    examples.append((vals, label))

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] XOR pattern: (Storm,High) and (Clear,Low) are delayed")
print("      (Storm,Low) and (Clear,High) are on time")
print("      Needs disjunction (OR) but hypothesis space only conjunctions")

SITUATION 3: XOR Pattern

[INPUT]
  E1: {'Weather': 'Storm', 'Congestion': 'High'} -> Positive
  E2: {'Weather': 'Clear', 'Congestion': 'Low'} -> Positive
  E3: {'Weather': 'Storm', 'Congestion': 'Low'} -> Negative
  E4: {'Weather': 'Clear', 'Congestion': 'High'} -> Negative

[FIND-S]
  Final S: {'Weather': '?', 'Congestion': '?'}

[CANDIDATE ELIMINATION]

  After E1: {'Weather': 'Storm', 'Congestion': 'High'} -> Positive
    S = {'Weather': 'Storm', 'Congestion': 'High'}
    G = [{'Weather': '?', 'Congestion': '?'}]

  After E2: {'Weather': 'Clear', 'Congestion': 'Low'} -> Positive
    S = {'Weather': '?', 'Congestion': '?'}
    G = [{'Weather': '?', 'Congestion': '?'}]

  After E3: {'Weather': 'Storm', 'Congestion': 'Low'} -> Negative
    S = {'Weather': '?', 'Congestion': '?'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] XOR pattern: (Storm,High) and (Clear,Low) are delayed
      (Storm,Low) and (Clear,High) are on time
      Needs disjunction (OR

In [7]:
print("="*70)
print("SITUATION 4: Missing Crucial Attribute")
print("="*70)

# Filter dataset for situation 4
sit4 = df[df['Sit'] == 4]
attrs = ['Airline', 'TimeOfDay']
examples = []

for _, row in sit4.iterrows():
    vals = [row['Airline'], row['TimeOfDay']]
    label = 'Positive' if row['Label'] == 'Delayed' else 'Negative'
    examples.append((vals, label))

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] True delay depends on missing attributes (Weather, Congestion, PreviousDelay)")
print("      No hypothesis using only Airline and TimeOfDay works")

SITUATION 4: Missing Crucial Attribute

[INPUT]
  E1: {'Airline': 'Delta', 'TimeOfDay': 'Evening'} -> Positive
  E2: {'Airline': 'Delta', 'TimeOfDay': 'Morning'} -> Negative
  E3: {'Airline': 'United', 'TimeOfDay': 'Evening'} -> Negative

[FIND-S]
  Final S: {'Airline': 'Delta', 'TimeOfDay': 'Evening'}

[CANDIDATE ELIMINATION]

  After E1: {'Airline': 'Delta', 'TimeOfDay': 'Evening'} -> Positive
    S = {'Airline': 'Delta', 'TimeOfDay': 'Evening'}
    G = [{'Airline': '?', 'TimeOfDay': '?'}]

  After E2: {'Airline': 'Delta', 'TimeOfDay': 'Morning'} -> Negative
    S = {'Airline': 'Delta', 'TimeOfDay': 'Evening'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] True delay depends on missing attributes (Weather, Congestion, PreviousDelay)
      No hypothesis using only Airline and TimeOfDay works


In [8]:
print("="*70)
print("SITUATION 5: Sequential Contradiction")
print("="*70)

# Filter dataset for situation 5
sit5 = df[df['Sit'] == 5]
attrs = ['Airline', 'Weather', 'PreviousDelay']
examples = []

for _, row in sit5.iterrows():
    vals = [row['Airline'], row['Weather'], row['PreviousDelay']]
    label = 'Positive' if row['Label'] == 'Delayed' else 'Negative'
    examples.append((vals, label))

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")
    if not G_state:
        print(f"    >>> EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] E4 contradicts E1 after algorithm updated boundaries")
print("      S covers negative, G cannot exclude without losing positives")

SITUATION 5: Sequential Contradiction

[INPUT]
  E1: {'Airline': 'Delta', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Positive
  E2: {'Airline': 'Delta', 'Weather': 'Rain', 'PreviousDelay': 'No'} -> Negative
  E3: {'Airline': 'United', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Positive
  E4: {'Airline': 'Delta', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Negative

[FIND-S]
  Final S: {'Airline': '?', 'Weather': 'Storm', 'PreviousDelay': 'Yes'}

[CANDIDATE ELIMINATION]

  After E1: {'Airline': 'Delta', 'Weather': 'Storm', 'PreviousDelay': 'Yes'} -> Positive
    S = {'Airline': 'Delta', 'Weather': 'Storm', 'PreviousDelay': 'Yes'}
    G = [{'Airline': '?', 'Weather': '?', 'PreviousDelay': '?'}]

  After E2: {'Airline': 'Delta', 'Weather': 'Rain', 'PreviousDelay': 'No'} -> Negative
    S = {'Airline': 'Delta', 'Weather': 'Storm', 'PreviousDelay': 'Yes'}
    G = []
    >>> VERSION SPACE EMPTY <<<
    >>> EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] E4 contradicts E1 after alg

In [10]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print("\nSituation 1: Identical diff labels → EMPTY")
print("Situation 2: Positive ⊏ Negative → EMPTY")
print("Situation 3: XOR pattern → EMPTY")
print("Situation 4: Missing attribute → EMPTY")
print("Situation 5: Sequential contradiction → EMPTY")

print("\n" + "="*70)
print("CONCLUSION: All 5 situations result in EMPTY VERSION SPACE")
print("="*70)



SUMMARY

Situation 1: Identical diff labels → EMPTY
Situation 2: Positive ⊏ Negative → EMPTY
Situation 3: XOR pattern → EMPTY
Situation 4: Missing attribute → EMPTY
Situation 5: Sequential contradiction → EMPTY

CONCLUSION: All 5 situations result in EMPTY VERSION SPACE
